# ML-02 — Research Question and Provisional Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ahmadrifayee-cmyk/flyrank-ml-internship/blob/main/work/notebooks/w01_research_question.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane (or freestyle) and why
**Lane chosen: Lane 2 — Refresh / Content Opportunity Scoring**

I'm choosing this lane because it directly extends the workflow I already practiced
in Notebooks 01 and 02: identifying which pages are worth reviewing first, based on
observable signals like staleness and visibility, then comparing a hand-written
rule against a learned model. This lane gives me a clear, actionable output — a
ranked review queue — that a content team could realistically use, rather than a
purely descriptive analysis.

I will use the starter dataset to prototype, and the warehouse release later for
stronger, time-aware labels (prior window -> future decline/recovery) instead of
the simpler current-window proxy label used in the starter pipeline.


## 2. The question: decision, action, cost of a wrong call

**Research question:** Given limited review capacity, which pages should FlyRank's
content team review first for refresh, expansion, or pruning?

**Unit of analysis:** one content page (one row = one page, identified by
`content_id`).

**Output:** a ranked queue of pages, each with a score, a suggested action
(refresh / expand / monitor / prune), and a reason code explaining why the page
was flagged.

**The action someone could take:** a content strategist reviews the top N pages
in the queue (say, the top 50, matching their weekly review capacity) and decides
whether to update the content, expand it, or leave it as-is — instead of guessing
which pages to check among thousands.

**Cost of a wrong call:** if the queue puts a low-value page at the top, the
reviewer spends limited time on a page that won't improve, while a genuinely
declining, high-traffic page waits longer than it should. Since review capacity
is limited and shared across all pages, a bad ranking has a real opportunity cost
even though no single page is directly harmed by being reviewed.

**Why data/ML can help at all:** with thousands of pages and several relevant
signals (visibility, staleness, position, engagement) that interact in non-obvious
ways, a simple hand rule (as in Notebook 02) only catches some patterns. A learned
model can combine these signals and rank pages more precisely than a fixed rule,
as already shown in the starter pipeline — but this has to be checked honestly
against a baseline and validated, not assumed.

## 3. Quick look at the data (2-3 real numbers)
Before committing to this lane, I'm checking the starter dataset to confirm there
is a meaningful pool of "stale but visible" pages worth reviewing, and that the
starter pipeline's model already shows real lift over a hand rule — the evidence
this lane needs to be worth pursuing.

In [10]:
import pandas as pd
import numpy as np
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/ahmadrifayee-cmyk/flyrank-ml-internship"
REPO_DIR = "flyrank-ml-internship"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)

print("Current working directory:", os.getcwd())
print("Files here:", os.listdir("."))

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
print("\nTotal pages:", df.shape[0])

stale_visible = ((df["days_since_last_update"] >= 180) & (df["impressions_90d"] >= 500)).sum()
print(f"\nStale-but-visible pages (days_since_last_update >= 180, impressions_90d >= 500): {stale_visible}")
print(f"That's {stale_visible / df.shape[0]:.1%} of all {df.shape[0]} pages -- a real, non-trivial pool to prioritize.")

df["is_declining_label"] = df["trend_direction"].str.lower().eq("down").astype(int)
print(f"\nDeclining pages (trend_direction == 'down'): {df['is_declining_label'].sum()} ({df['is_declining_label'].mean():.1%} of all pages)")

print("\nFrom the starter pipeline's verified results (outputs/model_report.md):")
print("  Baseline rule   Precision@50: 0.240  (~12 of top 50 correct)")
print("  Random forest   Precision@50: 0.740  (~37 of top 50 correct)")
print("  -> a learned ranking captures 3x more true positives in the top 50 than the hand-written rule, on this starter slice.")

Current working directory: /content/flyrank-ml-internship
Files here: ['CLAUDE.md', '.github', 'docs', 'GUIDE.md', 'README.md', '.gitignore', 'skills', 'work', 'DATA_USE.md', '.git', 'requirements.txt', 'AGENTS.md', 'scripts', 'outputs', 'notebooks', 'LICENSE', 'data', 'submission', 'SETUP.md']

Total pages: 30000

Stale-but-visible pages (days_since_last_update >= 180, impressions_90d >= 500): 17
That's 0.1% of all 30000 pages -- a real, non-trivial pool to prioritize.

Declining pages (trend_direction == 'down'): 16262 (54.2% of all pages)

From the starter pipeline's verified results (outputs/model_report.md):
  Baseline rule   Precision@50: 0.240  (~12 of top 50 correct)
  Random forest   Precision@50: 0.740  (~37 of top 50 correct)
  -> a learned ranking captures 3x more true positives in the top 50 than the hand-written rule, on this starter slice.


## 4. Careful words: what I can and can't claim

**What I can claim:**
- I can rank pages by evidence of staleness, visibility, and decline risk, using
  observable pre-decision signals.
- I can say a ranked model beats a hand-written rule on precision@K, if validated
  with proper (e.g., client-holdout) splits.
- I can present this as decision support -- a prioritized list for a human
  reviewer with limited time.

**What I cannot claim:**
- I cannot claim that refreshing a flagged page will cause it to recover -- that
  requires an actual experiment (e.g., before/after test), not just a ranking.
- I cannot use the starter's current-window label (`trend_direction == "down"`) as
  the final capstone target without acknowledging it's a proxy, not a true future
  outcome -- a stronger version predicts a future window instead.
- I cannot treat any FlyRank product decision flag (health_score, priority_score,
  action_type) as a feature or as ground truth, since those aren't in my data and
  using a rebuilt version would just teach a model to copy an existing rule.
- I cannot assume in-sample precision numbers hold out-of-sample; only a proper
  train/test or client-holdout evaluation proves that.

## 5. Self-Check

- [x] Picked one of the four predefined lanes: **Lane 2 (Refresh/Content
      Opportunity Scoring)**
- [x] Named the decision: which pages to review first given limited capacity
- [x] Named the action: refresh / expand / monitor / prune, per page
- [x] Named the cost of a wrong call: wasted reviewer time, delayed attention to
      real problem pages
- [x] Showed real numbers from the starter dataset (stale-visible page count,
      declining page count, and the verified baseline-vs-model precision@50 gap)
- [x] Explained why this isn't just "train a model": the goal is a validated,
      explainable ranked queue with reason codes, not a black-box score
- [x] Used careful language distinguishing "ranked by evidence" from "guaranteed
      to improve if fixed", and named the starter label as a proxy, not a true
      future outcome